# Agentic RAG over the LUMI documentation

This notebook builds a question-answering assistant for the LUMI supercomputer. It downloads the
latest LUMI documentation, indexes it, and lets an agent decide what to search for, read, and
search again before it answers — with links to the documentation pages it used.

## Models

Both models are called over an OpenAI-compatible API, so nothing heavy runs on your machine. The
default is [Aitta](https://aitta.csc.fi/public), CSC's inference service: put your key in `.env`
as `AITTA_API_KEY`. To use another provider (Ollama, vLLM, OpenAI, …), change `BASE_URL`,
`API_KEY` and the model names. The language model must support **tool calling**.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

load_dotenv()

BASE_URL = "https://aitta-api.csc.fi/openai/v1"
API_KEY = os.environ["AITTA_API_KEY"]
LLM_MODEL = "google/gemma-4-26B-A4B-it"
EMBEDDING_MODEL = "intfloat/multilingual-e5-large"

llm = ChatOpenAI(model=LLM_MODEL, base_url=BASE_URL, api_key=API_KEY, max_retries=5, timeout=300)


class E5Embeddings(OpenAIEmbeddings):
    """e5 models were trained with "query: " and "passage: " prefixes.
    Leaving them out does not raise an error; it just quietly makes search worse."""

    def embed_documents(self, texts, chunk_size=None, **kwargs):
        return super().embed_documents([f"passage: {t}" for t in texts], chunk_size, **kwargs)

    def embed_query(self, text, **kwargs):
        return super().embed_documents([f"query: {text}"], **kwargs)[0]


embeddings = E5Embeddings(
    model=EMBEDDING_MODEL,
    base_url=BASE_URL,
    api_key=API_KEY,
    check_embedding_ctx_length=False,  # send plain text; the server does its own tokenisation
    chunk_size=64,                     # texts sent per API request (a batch size, not a text length)
    max_retries=5,
    request_timeout=180,
)

## Warm-up

Models on a shared service are unloaded when nobody uses them, and loading one back can take a
minute or two. This cell wakes both models at the same time and waits until both have answered.

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor

started = time.perf_counter()
with ThreadPoolExecutor() as pool:
    llm_reply = pool.submit(llm.invoke, "Reply with just: OK")
    vector = pool.submit(embeddings.embed_query, "warm-up")
    print(f"LLM replied:              {llm_reply.result().content!r}")
    print(f"Embedding model replied:  a {len(vector.result())}-dimensional vector")
print(f"Both models ready after {time.perf_counter() - started:.0f}s")

## Configuration

The knowledge base is two public GitHub repositories: the
[LUMI user guide](https://docs.lumi-supercomputer.eu/) and the
[LUMI AI Guide](https://github.com/Lumi-supercomputer/LUMI-AI-Guide). To use your own documents,
point `DOC_SOURCES` at your own repositories of markdown files.

In [ ]:
CORPUS_DIR = "corpus"               # cloned documentation
PARENT_STORE_PATH = "parent_store"  # full sections, looked up by ID
QDRANT_DB_PATH = "qdrant_db"        # local vector database, no server needed
CHILD_COLLECTION = "lumi_docs"

DOC_SOURCES = [
    {   # Published as a website, so answers can link to the real pages
        "name": "lumi-userguide",
        "repo": "https://github.com/Lumi-supercomputer/lumi-userguide.git",
        "branch": "production",
        "docs_dir": "docs",
        "site_url": "https://docs.lumi-supercomputer.eu/",
    },
    {   # No website, so answers link to the file on GitHub
        "name": "LUMI-AI-Guide",
        "repo": "https://github.com/Lumi-supercomputer/LUMI-AI-Guide.git",
        "branch": "main",
        "docs_dir": "",
        "site_url": None,
    },
]

CHILD_CHUNK_SIZE = 800       # characters per searchable excerpt (4 characters are roughly 1 token)
CHILD_CHUNK_OVERLAP = 100
PARENT_MAX_CHARS = 8000      # longest section handed to the LLM in one go
DEFAULT_RETRIEVAL_K = 5      # excerpts per search
CHILD_CHUNK_SEPARATOR = "\n\n<CHILD_CHUNK_BOUNDARY>\n\n"

## Fetch the documentation

Documentation changes, so every run starts with a fresh copy. A shallow clone downloads only the
latest version and takes a few seconds. We remember which commit each repository is at, so the
slow part — indexing — is only redone when something has actually changed.

In [ ]:
import shutil
import subprocess
from pathlib import Path


def clone(source):
    """Shallow-clone one documentation repository and return the commit it is at."""
    dest = Path(CORPUS_DIR) / source["name"]
    shutil.rmtree(dest, ignore_errors=True)
    subprocess.run(
        ["git", "clone", "--quiet", "--depth", "1", "--branch", source["branch"], source["repo"], str(dest)],
        check=True,
    )
    return subprocess.run(
        ["git", "-C", str(dest), "rev-parse", "--short", "HEAD"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()


current_commits = {source["name"]: clone(source) for source in DOC_SOURCES}
for name, commit in current_commits.items():
    print(f"✓ {name} @ {commit}")

## Vector database

Qdrant runs inside this notebook and stores its data in a folder, so there is no server to set
up. We decide here whether the index on disk is still current: it is rebuilt if either
documentation repository has a new commit, or if the embedding model changed.

In [ ]:
import json
from qdrant_client import QdrantClient
from qdrant_client.http import models as qmodels
from langchain_qdrant import QdrantVectorStore

# The agent searches from several threads at once; access is serialised with a lock (see Tools).
client = QdrantClient(path=QDRANT_DB_PATH, force_disable_check_same_thread=True)

INDEX_STATE_FILE = Path(PARENT_STORE_PATH) / "index_state.json"
PARENTS_FILE = Path(PARENT_STORE_PATH) / "parents.json"

wanted_state = {**current_commits, "embedding_model": EMBEDDING_MODEL}
indexed_state = json.loads(INDEX_STATE_FILE.read_text()) if INDEX_STATE_FILE.exists() else None
needs_reindex = indexed_state != wanted_state or not client.collection_exists(CHILD_COLLECTION)

print("Index must be rebuilt." if needs_reindex else "Index is up to date.")

## Document Indexing

Each page is cut into **sections** at its `#` and `##` headings — these are the *parents*. Each
section is then cut into short overlapping **excerpts** — the *children*. Only the children are
embedded and searched, because short texts give sharper search results. When an excerpt looks
relevant but incomplete, the agent can fetch its whole parent section to read.

Every excerpt starts with its section's heading path (for example `Running jobs > Partitions`),
so an excerpt from the middle of a section still says what it is about.

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

section_splitter = MarkdownHeaderTextSplitter([("#", "h1"), ("##", "h2")])
excerpt_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHILD_CHUNK_SIZE, chunk_overlap=CHILD_CHUNK_OVERLAP
)


def page_url(source, rel_path):
    """Where a reader can open this page: the website if there is one, otherwise GitHub."""
    if source["site_url"]:
        # The website's address mirrors the file path: runjobs/partitions.md -> runjobs/partitions/
        parts = rel_path.with_suffix("").parts
        if parts[-1] == "index":
            parts = parts[:-1]
        return source["site_url"] + "".join(f"{part}/" for part in parts)
    return source["repo"].removesuffix(".git") + f"/blob/{source['branch']}/{rel_path.as_posix()}"


def load_sections():
    """Cut every markdown page into sections. Returns {parent_id: section}."""
    parents = {}
    for source in DOC_SOURCES:
        docs_dir = Path(CORPUS_DIR) / source["name"] / source["docs_dir"]
        for path in sorted(docs_dir.rglob("*.md")):
            rel_path = path.relative_to(docs_dir)
            if any(part.startswith(".") for part in rel_path.parts):  # skip .github/ and the like
                continue
            for section in section_splitter.split_text(path.read_text(encoding="utf-8")):
                if not section.page_content.strip():
                    continue
                parents[f"P{len(parents):04d}"] = {
                    "heading": " > ".join(section.metadata.values()) or rel_path.stem,
                    "url": page_url(source, rel_path),
                    "text": section.page_content,
                }
    return parents


def make_excerpts(parents):
    """Cut each section into short excerpts, each labelled with its heading and parent."""
    return [
        Document(
            page_content=f"{parent['heading']}\n\n{excerpt}",
            metadata={"parent_id": parent_id, "url": parent["url"]},
        )
        for parent_id, parent in parents.items()
        for excerpt in excerpt_splitter.split_text(parent["text"])
    ]


if needs_reindex:
    parents = load_sections()
    children = make_excerpts(parents)
    print(f"{len(parents)} sections -> {len(children)} excerpts to embed")

    if client.collection_exists(CHILD_COLLECTION):
        client.delete_collection(CHILD_COLLECTION)
    client.create_collection(
        CHILD_COLLECTION,
        vectors_config=qmodels.VectorParams(
            size=len(embeddings.embed_query("probe")),  # the vector length depends on the model
            distance=qmodels.Distance.COSINE,
        ),
    )
    child_vector_store = QdrantVectorStore(client=client, collection_name=CHILD_COLLECTION, embedding=embeddings)

    started, batch_size = time.perf_counter(), 256
    for start in range(0, len(children), batch_size):
        child_vector_store.add_documents(children[start:start + batch_size])
        done = min(start + batch_size, len(children))
        print(f"   embedded {done}/{len(children)} | {time.perf_counter() - started:.0f}s", end="\r")
    print(f"\n✓ Indexed in {time.perf_counter() - started:.0f}s")

    Path(PARENT_STORE_PATH).mkdir(exist_ok=True)
    PARENTS_FILE.write_text(json.dumps(parents, ensure_ascii=False), encoding="utf-8")
    INDEX_STATE_FILE.write_text(json.dumps(wanted_state, indent=2))  # written last: marks the index complete
else:
    child_vector_store = QdrantVectorStore(client=client, collection_name=CHILD_COLLECTION, embedding=embeddings)
    parents = json.loads(PARENTS_FILE.read_text(encoding="utf-8"))
    print(f"✓ Reusing index: {client.count(CHILD_COLLECTION).count} excerpts from {len(parents)} sections")

## Tools Definition

In [ ]:
import threading
from langchain_core.tools import tool

# Local Qdrant is not thread-safe, and the agent answers sub-questions in parallel threads.
_retrieval_lock = threading.Lock()


@tool
def search_child_chunks(query: str, limit: int = DEFAULT_RETRIEVAL_K) -> str:
    """Search the documentation for short excerpts related to the question.

    Use this as the first retrieval step. Each result has a Parent ID, the source URL and
    an excerpt that starts with its section heading. If an excerpt is relevant but
    incomplete, call retrieve_parent_chunks with its Parent ID to read the whole section.

    Args:
        query: Focused search query with concrete keywords from the question.
        limit: Maximum number of excerpts to return.
    """
    print("🔎", query)
    try:
        with _retrieval_lock:
            results = child_vector_store.similarity_search(query, k=limit)
        if not results:
            return "NO_RELEVANT_CHUNKS"
        return CHILD_CHUNK_SEPARATOR.join(
            f"Parent ID: {doc.metadata['parent_id']}\n"
            f"Source: {doc.metadata['url']}\n"
            f"Content: {doc.page_content.strip()}"
            for doc in results
        )
    except Exception as e:
        return f"RETRIEVAL_ERROR: {e}"


@tool
def retrieve_parent_chunks(parent_id: str) -> str:
    """Read the whole documentation section that a search result came from.

    Use this only after search_child_chunks returned a relevant but incomplete excerpt.

    Args:
        parent_id: Parent ID from a search_child_chunks result, for example P0042.
    """
    print("📄 reading section", parent_id)
    parent = parents.get(parent_id.strip())
    if parent is None:
        return "NO_PARENT_DOCUMENT"

    text, note = parent["text"].strip(), ""
    if len(text) > PARENT_MAX_CHARS:
        text, note = text[:PARENT_MAX_CHARS], f" (first {PARENT_MAX_CHARS} characters only)"

    return (
        f"Parent ID: {parent_id}{note}\n"
        f"Source: {parent['url']}\n"
        f"Section: {parent['heading']}\n"
        f"Content: {text}"
    )


llm_with_tools = llm.bind_tools([search_child_chunks, retrieve_parent_chunks])

## System prompts

In [ ]:
def get_conversation_summary_prompt() -> str:
    return """## Role
You are a compact memory manager for a retrieval-augmented chat assistant.

## Context
The input contains an existing rolling summary plus older user/assistant messages that will be removed from raw chat history.

## Instructions
- Merge the existing summary with the new older messages.
- Preserve context needed for future follow-up questions: topics, user preferences, important facts, unresolved questions, and referenced source URLs.
- Discard greetings, tool calls, tool outputs, formatting chatter, duplicate details, and resolved misunderstandings.
- Keep the summary compact: 30-70 words unless more detail is essential.

## Output
Return exactly one merged summary and nothing else.
Do not include labels such as "Updated summary:", "Previous summary:", or "New messages:".
Do not include both old and new summaries.
If there is no meaningful context, return an empty string.
"""

def get_rewrite_query_prompt() -> str:
    return """## Role
You are a query rewriting specialist for document retrieval in a RAG system.

## Instructions
- Rewrite the current query so it is clear, self-contained, and useful for retrieval.
- Use the conversation summary and recent conversation only to resolve vague follow-ups that refer to prior context.
- When an unresolved query and one or more user clarifications are provided, combine all of them into one self-contained retrieval query.
- If the query is a follow-up, integrate only the minimal context needed to make it self-contained.
- Preserve product names, file names, versions, acronyms, numbers, and technical terms exactly.
- If the user asks about a named topic, product, file, acronym, term, or concept, treat the question as clear even if it is new.
- Standalone named terms, acronyms, or concepts are valid retrieval queries; do not require prior conversation context.
- Split only truly separate information needs, with a maximum of 3 rewritten questions.

## Clarification Boundary
Mark the query unclear only when it depends on an unresolved reference such as "it", "that", "this file", or "the previous one".
Do not mark a query unclear because the topic was not mentioned earlier.
Do not ask the user whether a new acronym or term is a typo; preserve it and search for it.

## Constraints
Do not add facts, expand acronyms, invent context, or broaden the user's meaning.
"""

def get_orchestrator_prompt() -> str:
    return """## Role
You are a document-grounded research assistant for an agentic RAG system. Your job is to answer using retrieved document evidence, not general knowledge.

## Available Context
- Current user question
- Optional compressed context from prior retrieval steps
- Tools for searching short excerpts and loading the full section an excerpt came from

## Tool Guidance
- Search documents before answering unless compressed context already contains enough evidence.
- Use 'search_child_chunks' for missing or uncovered parts of the question.
- If searched or retrieved context is not useful, use the tools again with a different, simpler query or a more relevant parent section.
- Continue tool use until the available evidence is enough, tools stop adding useful information, or the operation limit is reached.
- Do not repeat search queries or parent IDs listed in compressed context.
- Do not retrieve the same parent ID twice.

## Response Framework
1. Check compressed context for already-known evidence and already-used searches or parents.
2. Search for missing evidence.
3. Retrieve parent sections only when excerpts are relevant but too fragmented.
4. Answer using the exact terms and scope in the retrieved evidence.
5. If evidence is incomplete, state the specific gap.

## Output
- Start directly with the substantive answer. Do not start with generic headings such as "Answer", "Final answer", or "Response".
- Provide the direct answer plus the key supporting details from retrieved evidence; avoid one-sentence fragments unless only one fact is available.
- Do not mention internal tool calls or reasoning.
- When sources exist, end with a Sources section in exactly this format:
  Sources:
  - https://example.org/page/
- Put each source URL on its own bullet line. Never write sources inline.
- Copy URLs exactly from the "Source:" lines of the retrieved results. Do not invent, shorten, or modify them.
"""

def get_fallback_response_prompt() -> str:
    return """## Role
You are a constrained evidence synthesizer for a retrieval-augmented assistant after the research loop reached its limit.

## Available Context
- Compressed Research Context from earlier retrieval steps
- Retrieved Data from current tool outputs

## Instructions
- Use only explicit facts from the provided context.
- Start directly with the substantive answer. Do not start with generic headings such as "Answer", "Final answer", or "Response".
- Prefer current Retrieved Data over compressed context if they conflict.
- If the answer is incomplete, mention only the missing parts that matter to the user query.
- Do not describe the retrieval process, limits, or internal reasoning.
- Be concise: answer in 1-3 short paragraphs or up to 5 bullets unless the user asks for detail.
- Provide the direct answer plus the key supporting details from retrieved evidence; avoid one-sentence fragments unless only one fact is available.
- End with a Sources section only when source URLs are explicitly present in the context.
- Use exactly this format:
  Sources:
  - https://example.org/page/
- Put each source URL on its own bullet line. Never write sources inline.
- Copy URLs exactly as they appear in the context. Do not invent, shorten, or modify them.
"""

def get_context_compression_prompt() -> str:
    return """## Role
You are a research context compressor for an agentic RAG system.

## Instructions
- Keep only facts relevant to answering the user question.
- Preserve exact names, figures, versions, technical terms, configuration details, and source URLs.
- Remove duplicates, tool chatter, search query wording, parent IDs, chunk IDs, and other internal identifiers.
- Organize findings by source. Each source section heading must be the exact source URL found in retrieved data.
- Add a Gaps section only for missing information relevant to the question.
- Target 400-600 words. If there is too much content, keep the most answer-critical facts.

## Output
Return only Markdown in this structure:
# Research Context Summary

## Focus
[Brief technical restatement of the question]

## Structured Findings
For each source, add a level-3 heading with its exact source URL and bullet the directly relevant facts below it.

## Gaps
- Missing or incomplete aspects
"""

def get_aggregation_prompt() -> str:
    return """## Role
You are a final-answer synthesizer for a retrieval-augmented assistant.

## Instructions
- Use only information present in the retrieved answers.
- Start directly with the substantive answer. Do not start with generic headings such as "Answer", "Final answer", or "Response".
- Preserve important names, numbers, versions, examples, and definitions.
- Do not expand acronyms or interpret terms unless the sources do it.
- If answers conflict, mention the conflict plainly.
- Be concise: answer in 1-3 short paragraphs or up to 5 bullets unless the user asks for detail.
- Provide the direct answer plus the key supporting details from retrieved evidence; avoid one-sentence fragments unless only one fact is available.
- End with a Sources section only when source URLs are explicitly present in the retrieved answers.
- Use exactly this format:
  Sources:
  - https://example.org/page/
- Put each source URL on its own bullet line, without duplicates. Never write sources inline.
- Copy URLs exactly as they appear in the retrieved answers. Do not invent, shorten, or modify them.
- If no useful information is available, say: "I couldn't find any information to answer your question in the available sources."
"""

## State Definitions

Define state schemas for managing conversation flow and agent execution.

In [ ]:
from langgraph.graph import MessagesState
from pydantic import BaseModel, Field
from typing import List, Annotated, Set
import operator

def accumulate_or_reset(existing: List[dict], new: List[dict]) -> List[dict]:
    if new and any(item.get('__reset__') for item in new):
        return []
    return existing + new

def set_union(a: Set[str], b: Set[str]) -> Set[str]:
    return a | b

def append_unique(existing: List[str], new: List[str]) -> List[str]:
    return list(dict.fromkeys(existing + new))

class State(MessagesState):
    """State for main agent graph"""
    questionIsClear: bool = False
    conversation_summary: str = ""
    originalQuery: str = ""
    pendingQuery: str = ""
    pendingClarifications: List[str] = []
    rewrittenQuestions: List[str] = []
    agent_answers: Annotated[List[dict], accumulate_or_reset] = []

class AgentState(MessagesState):
    """State for individual agent subgraph"""
    tool_call_count: Annotated[int, operator.add] = 0
    iteration_count: Annotated[int, operator.add] = 0
    question: str = ""
    question_index: int = 0
    context_summary: str = ""
    retrieval_keys: Annotated[Set[str], set_union] = set()
    retrieved_contexts: Annotated[List[str], append_unique] = []
    final_answer: str = ""
    agent_answers: List[dict] = []

class QueryAnalysis(BaseModel):
    is_clear: bool = Field(
        description="Indicates if the user's question is clear and answerable."
    )
    questions: List[str] = Field(
        description="List of rewritten, self-contained questions."
    )
    clarification_needed: str = Field(
        description="Explanation if the question is unclear."
    )



## Agent Limits and Token Utilities

Define execution limits and token counting utilities for controlling agent loop behavior.

In [ ]:
import tiktoken
from functools import lru_cache

MAX_TOOL_CALLS = 8
MAX_ITERATIONS = 10
BASE_TOKEN_THRESHOLD = 15000
TOKEN_GROWTH_FACTOR = 0.9

@lru_cache(maxsize=1)
def _get_token_encoding():
    try:
        return tiktoken.encoding_for_model("gpt-oss-120b")
    except Exception:
        try:
            return tiktoken.get_encoding("cl100k_base")
        except Exception:
            print("_get_token_encoding failed")
            return None

def estimate_context_tokens(messages: list) -> int:
    contents = [str(msg.content) for msg in messages if hasattr(msg, "content") and msg.content]
    encoding = _get_token_encoding()
    if encoding is None:
        return sum(max(1, len(content) // 4) for content in contents)
    return sum(len(encoding.encode(content)) for content in contents)



## Graph Nodes and Logic

Implement node functions that define the behavior of the agentic workflow.

What it does:
Core Nodes:

- summarize_history: Maintains rolling memory by summarizing older conversation, deleting summarized/tool messages, and keeping bounded raw chat history
- rewrite_query: Rewrites and optionally splits the user query into sub-questions while using the rolling summary and recent exchanges for context
- request_clarification: Interrupt point for human-in-the-loop when the query is ambiguous
- orchestrator: Main agent loop — invokes the LLM with tools, injects compressed context when available, and forces an initial retrieval call
- fallback_response: Generates a best-effort answer directly from all retrieved tool data when the agent loop exhausts its budget
- compress_context: Summarizes and compresses the current message history and prior context summary into a single block, appending a log of already-executed retrievals to prevent redundant calls; removes compressed messages from state
- collect_answer: Extracts the final assistant answer from the agent message history and packages it with its index for aggregation
- aggregate_answers: Sorts and merges all sub-question answers, then synthesizes them into a single coherent response matching the original user query


     

Routing Logic:

- route_after_rewrite: Routes to request_clarification if the query is unclear; otherwise uses Send to spawn parallel agent subgraphs — one per rewritten sub-question
- should_compress_context: Uses Command to conditionally route to compress_context (when token usage exceeds a dynamic threshold) or back to orchestrator; also tracks all retrieval keys across iterations to avoid repeated tool calls



In [ ]:
from langgraph.types import Send, Command
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, RemoveMessage, ToolMessage
from typing import Literal, Set

MAIN_HISTORY_MESSAGES_TO_KEEP = 4
if MAIN_HISTORY_MESSAGES_TO_KEEP < 2:
    raise ValueError("MAIN_HISTORY_MESSAGES_TO_KEEP must be at least 2.")
PRE_ANSWER_HISTORY_MESSAGES_TO_KEEP = max(MAIN_HISTORY_MESSAGES_TO_KEEP - 1, 0)

def _is_plain_conversation_message(msg) -> bool:
    return (
        isinstance(msg, (HumanMessage, AIMessage))
        and not getattr(msg, "tool_calls", None)
        and not getattr(msg, "name", None)
    )

def _name_internal_message(message, name):
    """Tag a subgraph-only message so it is not treated as chat history."""
    return message.model_copy(update={"name": name})

def _retrieval_contexts(messages) -> list[str]:
    contexts = []
    ignored_prefixes = (
        "NO_RELEVANT_CHUNKS",
        "NO_PARENT_DOCUMENT",
        "RETRIEVAL_ERROR:",
        "PARENT_RETRIEVAL_ERROR:",
    )
    for message in messages:
        if not isinstance(message, ToolMessage):
            continue
        content = str(message.content).strip()
        if content and not content.startswith(ignored_prefixes):
            parts = content.split(CHILD_CHUNK_SEPARATOR) if message.name == "search_child_chunks" else [content]
            contexts.extend(part for part in parts if part)
    return list(dict.fromkeys(contexts))

def _format_conversation(messages) -> str:
    lines = []
    for msg in messages:
        role = "User" if isinstance(msg, HumanMessage) else "Assistant"
        lines.append(f"{role}: {msg.content}")
    return "\n".join(lines)

def _remove_messages_not_in(messages, keep_ids):
    removals = []
    for msg in messages:
        msg_id = getattr(msg, "id", None)
        if isinstance(msg, SystemMessage) or not msg_id:
            continue
        if msg_id not in keep_ids:
            removals.append(RemoveMessage(id=msg_id))
    return removals

def _recent_conversation(messages, pending_query="") -> list:
    """Return recent context before the current user message."""
    plain_messages = [msg for msg in messages if _is_plain_conversation_message(msg)]
    recent_messages = plain_messages[:-1]

    if pending_query:
        for index in range(len(recent_messages) - 1, -1, -1):
            msg = recent_messages[index]
            if isinstance(msg, HumanMessage) and str(msg.content).strip() == pending_query:
                return recent_messages[:index]

    return recent_messages

def summarize_history(state: State):
    """Roll older chat into a summary and keep only bounded raw history."""
    messages = state.get("messages", [])
    updates = {"agent_answers": [{"__reset__": True}]}

    if not messages:
        return updates

    plain_messages = [msg for msg in messages if _is_plain_conversation_message(msg)]
    keep_count = PRE_ANSWER_HISTORY_MESSAGES_TO_KEEP
    messages_to_summarize = plain_messages[:-keep_count] if len(plain_messages) > keep_count else []
    keep_ids = {getattr(msg, "id", None) for msg in plain_messages[-keep_count:]}
    keep_ids.discard(None)

    removals = _remove_messages_not_in(messages, keep_ids)
    if removals:
        updates["messages"] = removals

    if not messages_to_summarize:
        return updates

    existing_summary = state.get("conversation_summary", "").strip()
    conversation = "Existing summary:\n"
    conversation += f"{existing_summary or '(none)'}\n\n"
    conversation += "New messages to merge into the summary:\n"
    conversation += _format_conversation(messages_to_summarize)

    summary_response = llm.invoke([
        SystemMessage(content=get_conversation_summary_prompt()),
        HumanMessage(content=conversation),
    ])
    updates["conversation_summary"] = summary_response.content.strip()
    return updates

def rewrite_query(state: State):
    """Analyzes user query and rewrites it for clarity, optionally using conversation context."""
    last_message = state["messages"][-1]
    current_query = str(last_message.content).strip()
    conversation_summary = state.get("conversation_summary", "").strip()
    pending_query = state.get("pendingQuery", "").strip()
    pending_clarifications = state.get("pendingClarifications", [])
    recent_messages = _recent_conversation(state["messages"], pending_query)

    context_parts = []
    if conversation_summary:
        context_parts.append(f"Conversation Summary:\n{conversation_summary}")
    if recent_messages:
        context_parts.append(f"Recent Conversation:\n{_format_conversation(recent_messages)}")

    if pending_query:
        clarifications = [*pending_clarifications, current_query]
        clarification_text = "\n".join(
            f"{index}. {value}" for index, value in enumerate(clarifications, start=1)
        )
        context_parts.append(
            f"Unresolved User Query:\n{pending_query}\n\n"
            f"User Clarifications:\n{clarification_text}"
        )
        original_query = f"{pending_query}\nClarifications:\n{clarification_text}"
    else:
        clarifications = []
        context_parts.append(f"User Query:\n{current_query}")
        original_query = current_query

    context_section = "\n\n".join(context_parts)
    llm_with_structure = llm.with_structured_output(QueryAnalysis, method="function_calling")
    response = llm_with_structure.invoke([SystemMessage(content=get_rewrite_query_prompt()), HumanMessage(content=context_section)])
    clarification_message_update = (
        [_name_internal_message(last_message, "clarification_response")]
        if pending_query else []
    )

    if response.questions and response.is_clear:
        return {
            "questionIsClear": True,
            "originalQuery": original_query,
            "pendingQuery": "",
            "pendingClarifications": [],
            "rewrittenQuestions": response.questions,
            "messages": clarification_message_update,
        }

    clarification = response.clarification_needed if response.clarification_needed and len(response.clarification_needed.strip()) > 10 else "I need more information to understand your question."
    return {
        "questionIsClear": False,
        "originalQuery": "",
        "pendingQuery": pending_query or current_query,
        "pendingClarifications": clarifications,
        "rewrittenQuestions": [],
        "messages": clarification_message_update + [
            AIMessage(content=clarification, name="clarification")
        ],
    }

def request_clarification(state: State):
    """Placeholder node for human-in-the-loop interruption."""
    return {}

def route_after_rewrite(state: State) -> Literal["request_clarification", "agent"]:
    if not state.get("questionIsClear", False):
        return "request_clarification"
    return [
        Send("agent", {"question": query, "question_index": idx, "messages": []})
        for idx, query in enumerate(state["rewrittenQuestions"])
    ]

def orchestrator(state: AgentState):
    """Main agent logic: decides when to search, retrieve, and answer."""
    context_summary = state.get("context_summary", "").strip()
    sys_msg = SystemMessage(content=get_orchestrator_prompt())
    summary_injection = (
        [HumanMessage(content=f"[COMPRESSED CONTEXT FROM PRIOR RESEARCH]\n\n{context_summary}")]
        if context_summary else []
    )

    if not state.get("messages"):
        human_msg = HumanMessage(content=state["question"], name="agent_question")
        force_search = HumanMessage(content="YOU MUST CALL 'search_child_chunks' AS THE FIRST STEP TO ANSWER THIS QUESTION.")
        response = llm_with_tools.invoke([sys_msg] + summary_injection + [human_msg, force_search])
        response = _name_internal_message(response, "agent_response")
        return {"messages": [human_msg, response], "tool_call_count": len(response.tool_calls or []), "iteration_count": 1}

    response = llm_with_tools.invoke([sys_msg] + summary_injection + state["messages"])
    response = _name_internal_message(response, "agent_response")
    tool_calls = response.tool_calls if hasattr(response, "tool_calls") else []
    return {"messages": [response], "tool_call_count": len(tool_calls) if tool_calls else 0, "iteration_count": 1}

def route_after_orchestrator_call(state: AgentState) -> Literal["tools", "fallback_response", "collect_answer"]:
    iteration = state.get("iteration_count", 0)
    tool_count = state.get("tool_call_count", 0)

    last_message = state["messages"][-1]
    tool_calls = getattr(last_message, "tool_calls", None) or []

    if not tool_calls:
        return "collect_answer"

    # Accept a final answer at the iteration boundary, but do not execute
    # tool calls that would exceed the configured research budget.
    if iteration >= MAX_ITERATIONS or tool_count > MAX_TOOL_CALLS:
        return "fallback_response"

    return "tools"

def fallback_response(state: AgentState):
    """Generate a best-effort answer using all retrieved information when limits are reached."""
    seen = set()
    unique_contents = []
    for m in state["messages"]:
        if isinstance(m, ToolMessage) and m.content not in seen:
            unique_contents.append(m.content)
            seen.add(m.content)

    context_summary = state.get("context_summary", "").strip()

    context_parts = []
    if context_summary:
        context_parts.append(f"## Compressed Research Context (from prior iterations)\n\n{context_summary}")
    if unique_contents:
        context_parts.append(
            "## Retrieved Data (current iteration)\n\n" +
            "\n\n".join(f"--- DATA SOURCE {i} ---\n{content}" for i, content in enumerate(unique_contents, 1))
        )

    context_text = "\n\n".join(context_parts) if context_parts else "No data was retrieved from the documents."
    prompt_content = (
        f"USER QUERY: {state.get('question')}\n\n"
        f"{context_text}\n\n"
        f"INSTRUCTION:\nProvide the best possible answer using only the data above."
    )
    response = llm.invoke([SystemMessage(content=get_fallback_response_prompt()), HumanMessage(content=prompt_content)])
    response = _name_internal_message(response, "agent_response")
    return {"messages": [response]}

def should_compress_context(state: AgentState) -> Command[Literal["compress_context", "orchestrator"]]:
    messages = state["messages"]

    new_ids: Set[str] = set()
    for msg in reversed(messages):
        if isinstance(msg, AIMessage) and getattr(msg, "tool_calls", None):
            for tc in msg.tool_calls:
                if tc["name"] == "retrieve_parent_chunks":
                    raw = tc["args"].get("parent_id") or tc["args"].get("id") or tc["args"].get("ids") or []
                    child = tc["args"].get("child_id") or ""
                    if isinstance(raw, str):
                        new_ids.add(f"parent::{raw}::{child}")
                    else:
                        new_ids.update(f"parent::{r}::{child}" for r in raw)

                elif tc["name"] == "search_child_chunks":
                    query = tc["args"].get("query", "")
                    if query:
                        new_ids.add(f"search::{query}")
            break

    updated_ids = state.get("retrieval_keys", set()) | new_ids

    current_token_messages = estimate_context_tokens(messages)
    current_token_summary = estimate_context_tokens([HumanMessage(content=state.get("context_summary", ""))])
    current_tokens = current_token_messages + current_token_summary
    max_allowed = BASE_TOKEN_THRESHOLD + int(current_token_summary * TOKEN_GROWTH_FACTOR)

    goto = "compress_context" if current_tokens > max_allowed else "orchestrator"
    return Command(
        update={
            "retrieval_keys": updated_ids,
            "retrieved_contexts": _retrieval_contexts(messages),
        },
        goto=goto,
    )

def compress_context(state: AgentState):
    """Compress retrieval history into a concise summary for future iterations."""
    messages = state["messages"]
    existing_summary = state.get("context_summary", "").strip()

    if not messages:
        return {}

    conversation_text = f"USER QUESTION:\n{state.get('question')}\n\nConversation to compress:\n\n"
    if existing_summary:
        conversation_text += f"[PRIOR COMPRESSED CONTEXT]\n{existing_summary}\n\n"

    for msg in messages[1:]:
        if isinstance(msg, AIMessage):
            tool_calls_info = ""
            if getattr(msg, "tool_calls", None):
                calls = ", ".join(f"{tc['name']}({tc['args']})" for tc in msg.tool_calls)
                tool_calls_info = f" | Tool calls: {calls}"
            conversation_text += f"[ASSISTANT{tool_calls_info}]\n{msg.content or '(tool call only)'}\n\n"
        elif isinstance(msg, ToolMessage):
            tool_name = getattr(msg, "name", "tool")
            conversation_text += f"[TOOL RESULT — {tool_name}]\n{msg.content}\n\n"

    summary_response = llm.invoke([SystemMessage(content=get_context_compression_prompt()), HumanMessage(content=conversation_text)])
    new_summary = summary_response.content

    retrieved_ids: Set[str] = state.get("retrieval_keys", set())
    if retrieved_ids:
        parent_ids = sorted(r for r in retrieved_ids if r.startswith("parent::"))
        search_queries = sorted(r.replace("search::", "") for r in retrieved_ids if r.startswith("search::"))

        block = "\n\n---\n**Already executed (do NOT repeat):**\n"
        if parent_ids:
            pretty = []
            for p in parent_ids:
                pid, _, cid = p[len("parent::"):].partition("::")
                pretty.append(f"- {pid}" + (f" (child {cid})" if cid else ""))
            block += "Parent chunks retrieved:\n" + "\n".join(pretty) + "\n"
        if search_queries:
            block += "Search queries already run:\n" + "\n".join(f"- {q}" for q in search_queries) + "\n"
        new_summary += block

    return {"context_summary": new_summary, "messages": [RemoveMessage(id=m.id) for m in messages[1:]]}

def collect_answer(state: AgentState):
    last_message = state["messages"][-1]
    is_valid = isinstance(last_message, AIMessage) and last_message.content and not last_message.tool_calls
    answer = last_message.content if is_valid else "Unable to generate an answer."
    return {
        "final_answer": answer,
        "agent_answers": [{
            "index": state["question_index"],
            "question": state["question"],
            "answer": answer,
            "contexts": state.get("retrieved_contexts", []),
        }]
    }

def aggregate_answers(state: State):
    messages = state.get("messages", [])
    plain_messages = [msg for msg in messages if _is_plain_conversation_message(msg)]
    keep_ids = {getattr(msg, "id", None) for msg in plain_messages[-PRE_ANSWER_HISTORY_MESSAGES_TO_KEEP:]}
    keep_ids.discard(None)
    removals = _remove_messages_not_in(messages, keep_ids)

    if not state.get("agent_answers"):
        return {"messages": removals + [AIMessage(content="No answers were generated.")]}

    sorted_answers = sorted(state["agent_answers"], key=lambda x: x["index"])

    formatted_answers = ""
    for i, ans in enumerate(sorted_answers, start=1):
        formatted_answers += (f"\nRetrieved response {i}:\n"f"{ans['answer']}\n")

    user_message = HumanMessage(content=f"""Original user question: {state["originalQuery"]}\nRetrieved answers:{formatted_answers}""")
    synthesis_response = llm.invoke([SystemMessage(content=get_aggregation_prompt()), user_message])
    return {"messages": removals + [AIMessage(content=synthesis_response.content)]}



## LangGraph Construction
Construct the agentic workflow using LangGraph's state machine.

In [ ]:
from langgraph.graph import START, END, StateGraph
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import InMemorySaver
from IPython.display import Image, display

# Initialize checkpointer
checkpointer = InMemorySaver()

# Build agent subgraph (handles individual questions)
agent_builder = StateGraph(AgentState)
agent_builder.add_node(orchestrator)
agent_builder.add_node("tools", ToolNode([search_child_chunks, retrieve_parent_chunks]))
agent_builder.add_node(compress_context)
agent_builder.add_node(fallback_response)
agent_builder.add_node(should_compress_context)
agent_builder.add_node(collect_answer)

agent_builder.add_edge(START, "orchestrator")
agent_builder.add_conditional_edges("orchestrator", route_after_orchestrator_call, {"tools": "tools", "fallback_response": "fallback_response", "collect_answer": "collect_answer"})
agent_builder.add_edge("tools", "should_compress_context")
agent_builder.add_edge("compress_context", "orchestrator")
agent_builder.add_edge("fallback_response", "collect_answer")
agent_builder.add_edge("collect_answer", END)
agent_subgraph = agent_builder.compile()

# Build main graph (orchestrates workflow)
graph_builder = StateGraph(State)
graph_builder.add_node(summarize_history)
graph_builder.add_node(rewrite_query)
graph_builder.add_node(request_clarification)
graph_builder.add_node("agent", agent_subgraph)
graph_builder.add_node(aggregate_answers)

graph_builder.add_edge(START, "summarize_history")
graph_builder.add_edge("summarize_history", "rewrite_query")
graph_builder.add_conditional_edges("rewrite_query", route_after_rewrite)
graph_builder.add_edge("request_clarification", "rewrite_query")
graph_builder.add_edge(["agent"], "aggregate_answers")
graph_builder.add_edge("aggregate_answers", END)

# Compile graph
agent_graph = graph_builder.compile(checkpointer=checkpointer, interrupt_before=["request_clarification"])

# Drawing the picture uses the mermaid.ink web service, which can be offline or rate-limited.
try:
    display(Image(agent_graph.get_graph(xray=True).draw_mermaid_png()))
except Exception as e:
    print(f"Could not render the graph image ({type(e).__name__}). Mermaid source instead:\n")
    print(agent_graph.get_graph(xray=True).draw_mermaid())
print("✓ Agent graph compiled successfully.")

## Observability

In [ ]:
import os
from langfuse import get_client
from langfuse.langchain import CallbackHandler

#LANGFUSE_ENABLED = os.environ.get("LANGFUSE_ENABLED", "false").lower() == "true"
LANGFUSE_ENABLED = False
langfuse_handler = None
langfuse = None

if LANGFUSE_ENABLED:
    langfuse = get_client()
    try:
        if langfuse.auth_check():
            print("Langfuse client is authenticated and ready!")
            langfuse_handler = CallbackHandler()
        else:
            print("Authentication failed. Please check your credentials and host.")
    except Exception as e:
        print(f"Error during Langfuse authentication: {e}")
else:
    print("Langfuse tracing disabled. Set LANGFUSE_ENABLED=true to enable it.")



## Ask a question

`ask()` sends one message in an ongoing conversation, so follow-up questions can refer to earlier
answers. If the agent finds a question too vague, it replies with a clarifying question instead
— answer it with another `ask(...)`. Run `new_conversation()` to start over.

In [ ]:
import uuid
from langchain_core.messages import HumanMessage



def new_conversation():
    global config
    config = {"configurable": {"thread_id": str(uuid.uuid4())}, "recursion_limit": 50}
    if langfuse_handler:
        config["callbacks"] = [langfuse_handler]


def ask(message: str):
    if agent_graph.get_state(config).next:  # paused, waiting for an answer to a clarifying question
        agent_graph.update_state(config, {"messages": [HumanMessage(content=message)]})
        result = agent_graph.invoke(None, config)
    else:
        result = agent_graph.invoke({"messages": [HumanMessage(content=message)]}, config)
    print(result["messages"][-1].content)


new_conversation()
ask("I need 2 full GPU nodes for 3 days. Which partition should I use?")

In [ ]:
# A follow-up: "that" only makes sense because the agent remembers the conversation
ask("How many GPU hours will that use?")

## Gradio Interface

*Disabled for now — the code below is kept as a markdown cell and will become a chat UI in a later version.*

import gradio as gr
import uuid
import json
import re
from langchain_core.messages import HumanMessage, AIMessageChunk, ToolMessage
import logging
# Suppress OTel "Failed to detach context" warning caused by generator/context interaction.
# Tracing is unaffected.
# Known bug: https://github.com/open-telemetry/opentelemetry-python/issues/2606
class _SuppressOtelDetachWarning(logging.Filter):
    def filter(self, record):
        return "Failed to detach context" not in record.getMessage()

logging.getLogger("opentelemetry.context").addFilter(_SuppressOtelDetachWarning())

# ── Constants ────────────────────────────────────────────────────────────────

SYSTEM_NODES  = {"summarize_history", "rewrite_query"}
FINAL_RESPONSE_NODES = {"aggregate_answers"}

SYSTEM_NODE_CONFIG = {
    "rewrite_query":     {"title": "🔍 Query Analysis & Rewriting"},
    "summarize_history": {"title": "📋 Chat History Summary"},
}

# ── Helpers ───────────────────────────────────────────────────────────────────

def create_thread_id():
    return {"configurable": {"thread_id": str(uuid.uuid4())}, "recursion_limit": 50}

def clear_session():
    global config
    agent_graph.checkpointer.delete_thread(config["configurable"]["thread_id"])
    config = create_thread_id()
    if langfuse is not None:
        try:
            langfuse.flush()
        except Exception:
            pass

def make_message(content, *, title=None, node=None):
    """Build a Gradio chat message dict, with optional collapsible metadata."""
    msg = {"role": "assistant", "content": content}
    if title or node:
        msg["metadata"] = {k: v for k, v in {"title": title, "node": node}.items() if v}
    return msg

def find_msg_idx(messages, node):
    """Return index of first message whose metadata.node matches, or None."""
    return next(
        (i for i, m in enumerate(messages) if m.get("metadata", {}).get("node") == node),
        None
    )

def parse_rewrite_json(buffer):
    """Try to parse structured JSON from rewrite_query buffer. Returns dict or None."""
    match = re.search(r'\{.*\}', buffer, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group())
    except Exception:
        return None

def format_rewrite_content(buffer):
    """Convert rewrite_query JSON buffer into readable markdown."""
    data = parse_rewrite_json(buffer)
    if not data:
        return "⏳ Analyzing query..."

    if data.get("is_clear"):
        lines = ["✅ **Query is clear**"]
        if data.get("questions"):
            lines += ["\n**Rewritten queries:**"] + [f"- {q}" for q in data["questions"]]
    else:
        lines = ["❓ **Query is unclear**"]
        clarification = data.get("clarification_needed", "")
        if clarification and clarification.strip().lower() != "no":
            lines.append(f"\nClarification needed: *{clarification}*")

    return "\n".join(lines)

# ── Observability ───────────────────────────────────────────────────────────────
# If the observability cell was skipped, default to no tracing
if "langfuse_handler" not in dir():
    langfuse_handler = None
if "langfuse" not in dir():
    langfuse = None

# ── Streaming ─────────────────────────────────────────────────────────────────

def chat(message, history):
    current_state = agent_graph.get_state(config)

    if current_state.next:
        agent_graph.update_state(config, {"messages": [HumanMessage(content=message.strip())]})
        stream_input = None
    else:
        stream_input = {"messages": [HumanMessage(content=message.strip())]}

    # Add Langfuse handler if observability is enabled
    run_config = {**config, **({"callbacks": [langfuse_handler]} if langfuse_handler else {})}

    response_messages = []
    active_tool_calls  = {}
    system_node_buffer = {}

    for chunk, metadata in agent_graph.stream(stream_input, config=run_config, stream_mode="messages"):
        node = metadata.get("langgraph_node", "")

        # ── System nodes (summarize_history, rewrite_query) ──────────────────
        if node in SYSTEM_NODES and isinstance(chunk, AIMessageChunk) and chunk.content:
            system_node_buffer[node] = system_node_buffer.get(node, "") + chunk.content
            buffer = system_node_buffer[node]
            idx    = find_msg_idx(response_messages, node)
            title  = SYSTEM_NODE_CONFIG[node]["title"]

            content = format_rewrite_content(buffer) if node == "rewrite_query" else buffer

            if idx is None:
                response_messages.append(make_message(content, title=title, node=node))
            else:
                response_messages[idx]["content"] = content

            # rewrite_query: surface clarification as a plain (non-collapsible) message
            if node == "rewrite_query":
                data = parse_rewrite_json(buffer) or {}
                clarification = data.get("clarification_needed", "")
                if not data.get("is_clear") and clarification.strip().lower() not in ("", "no"):
                    cidx = find_msg_idx(response_messages, "clarification")
                    if cidx is None:
                        response_messages.append(make_message(clarification, node="clarification"))
                    else:
                        response_messages[cidx]["content"] = clarification

            yield response_messages
            continue

        # ── Tool calls ────────────────────────────────────────────────────────
        if hasattr(chunk, "tool_calls") and chunk.tool_calls:
            for tc in chunk.tool_calls:
                if tc.get("id") and tc["id"] not in active_tool_calls:
                    response_messages.append(make_message(f"Running `{tc['name']}`...", title=f"🛠️ {tc['name']}"))
                    active_tool_calls[tc["id"]] = len(response_messages) - 1

        # ── Tool results ──────────────────────────────────────────────────────
        elif isinstance(chunk, ToolMessage):
            if (idx := active_tool_calls.get(chunk.tool_call_id)) is not None:
                preview = str(chunk.content)[:300]
                suffix  = "\n..." if len(str(chunk.content)) > 300 else ""
                response_messages[idx]["content"] = f"```\n{preview}{suffix}\n```"

        # ── LLM tokens (final response) ───────────────────────────────────────
        elif isinstance(chunk, AIMessageChunk) and chunk.content and node in FINAL_RESPONSE_NODES:
            last = response_messages[-1] if response_messages else None
            if not (last and last.get("role") == "assistant" and "metadata" not in last):
                response_messages.append(make_message(""))
            response_messages[-1]["content"] += chunk.content

        else:
            continue

        yield response_messages

# ── App ───────────────────────────────────────────────────────────────────────

config = create_thread_id()

with gr.Blocks() as demo:
    chatbot = gr.Chatbot(
        height=720,
        placeholder="<strong>Ask me anything!</strong><br><em>I'll search, reason, and act to give you the best answer :)</em>",
        show_label=False
    )
    chatbot.clear(clear_session)
    gr.ChatInterface(fn=chat, chatbot=chatbot)

print("\nLaunching application...")
demo.launch(theme=gr.themes.Citrus())

